# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tushar-sharma001/Flyrank-Ml-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub scikit-learn
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from huggingface_hub import HfApi

TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

MONTH = "2026-03"
BASE = "hf://datasets/FlyRank/internship-warehouse"

api = HfApi()
all_files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=TOKEN)
fact_files = [f for f in all_files if "fact_content_daily_performance" in f and f"month={MONTH}" in f]
next_month_files = [f for f in all_files if "fact_content_daily_performance" in f and "month=2026-04" in f]
dim_content_files = [f for f in all_files if "dim_content" in f.lower() and f.endswith(".parquet")]

FACT = [f"{BASE}/{f}" for f in fact_files]
FACT_NEXT = [f"{BASE}/{f}" for f in next_month_files]
DIM_CONTENT = [f"{BASE}/{f}" for f in dim_content_files]
FACT_STR = "[" + ", ".join(f"'{p}'" for p in FACT) + "]"
FACT_TWO_MONTHS_STR = "[" + ", ".join(f"'{p}'" for p in FACT + FACT_NEXT) + "]"
DIM_CONTENT_STR = "[" + ", ".join(f"'{p}'" for p in DIM_CONTENT) + "]"

# rebuild feat, train, test, rf, rf_proba exactly like the capstone notebook -- this notebook
# should stand on its own, not silently depend on another notebook's runtime state
df_content = con.sql(f"SELECT content_hash_id, content_created_date FROM read_parquet({DIM_CONTENT_STR})").df()

smoothed = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id, report_date, gsc_impressions,
        AVG(gsc_impressions) OVER (
            PARTITION BY content_hash_id ORDER BY report_date
            ROWS BETWEEN 89 PRECEDING AND CURRENT ROW
        ) AS prior90_avg_daily,
        AVG(gsc_impressions) OVER (
            PARTITION BY content_hash_id ORDER BY report_date
            ROWS BETWEEN 25 FOLLOWING AND 35 FOLLOWING
        ) AS future_window_avg_daily
    FROM read_parquet({FACT_TWO_MONTHS_STR})
""").df()
smoothed["report_date"] = pd.to_datetime(smoothed["report_date"])
smoothed_march = smoothed[smoothed["report_date"] < "2026-04-01"].dropna(subset=["future_window_avg_daily"])
decision_rows = smoothed_march.sort_values("report_date").groupby("content_hash_id").tail(1).copy()

MIN_DAILY_VOLUME = 10
DECLINE_THRESHOLD = 0.90
decision_rows["future_decline_label"] = (
    (decision_rows["future_window_avg_daily"] < decision_rows["prior90_avg_daily"] * DECLINE_THRESHOLD)
    & (decision_rows["prior90_avg_daily"] >= MIN_DAILY_VOLUME)
).astype(int)
label_by_content = decision_rows[["content_hash_id", "future_decline_label"]]

daily = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position
    FROM read_parquet({FACT_STR})
""").df()
daily["report_date"] = pd.to_datetime(daily["report_date"])
daily = daily.merge(df_content, on="content_hash_id", how="left")
daily["content_created_date"] = pd.to_datetime(daily["content_created_date"])
daily["days_since_update"] = (daily["report_date"] - daily["content_created_date"]).dt.days

feat = daily.groupby(["content_hash_id", "client_hash_id"]).agg(
    impressions_prior90=("gsc_impressions", "sum"),
    clicks_prior90=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    days_since_update=("days_since_update", "max"),
).reset_index()
feat["ctr_prior90"] = feat["clicks_prior90"] / feat["impressions_prior90"].replace(0, np.nan)
feat = feat.merge(label_by_content, on="content_hash_id", how="inner").dropna()

np.random.seed(42)
clients = feat["client_hash_id"].unique()
np.random.shuffle(clients)
split_point = int(len(clients) * 0.8)
train_clients, test_clients = clients[:split_point], clients[split_point:]
train = feat[feat["client_hash_id"].isin(train_clients)]
test = feat[feat["client_hash_id"].isin(test_clients)]

from sklearn.ensemble import RandomForestClassifier
feature_cols = ["impressions_prior90", "clicks_prior90", "avg_position", "days_since_update", "ctr_prior90"]
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(train[feature_cols], train["future_decline_label"])
test = test.copy()
test["rf_proba"] = rf.predict_proba(test[feature_cols])[:, 1]

print("rows ready for the playbook:", len(test))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows ready for the playbook: 32726


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue, with one of three reason codes per row -- readable by a human without
touching the model:

- zero_click_risk: real impressions but no clicks -- the model's own known blind
  spot, so this gets flagged regardless of score.
- model_and_rule_agree: high Random Forest score AND the transparent
  stale_but_visible rule also flags it -- two independent signals pointing the
  same way, highest confidence.
- model_only: high Random Forest score, but the simple rule doesn't flag it --
  still worth a look, lower confidence than the agreement case.

In [8]:
test["rule_flag"] = ((test["days_since_update"] >= 180) & (test["impressions_prior90"] >= 500)).astype(int)
test["zero_click_flag"] = ((test["impressions_prior90"] >= 200) & (test["clicks_prior90"] == 0)).astype(int)

def reason_code(row):
    if row["zero_click_flag"] == 1:
        return "zero_click_risk"
    if row["rf_proba"] >= 0.5 and row["rule_flag"] == 1:
        return "model_and_rule_agree"
    if row["rf_proba"] >= 0.5:
        return "model_only"
    return "no_action_needed"

test["reason_code"] = test.apply(reason_code, axis=1)
test["action"] = test["reason_code"].apply(lambda r: "review_for_refresh" if r != "no_action_needed" else "monitor")

queue = test.sort_values("rf_proba", ascending=False)
print(queue["reason_code"].value_counts())
queue[["content_hash_id", "client_hash_id", "rf_proba", "reason_code", "action"] + feature_cols].head(10)

reason_code
no_action_needed        16339
model_only               8672
zero_click_risk          4772
model_and_rule_agree     2943
Name: count, dtype: int64


,content_hash_id,client_hash_id,rf_proba,reason_code,action,impressions_prior90,clicks_prior90,avg_position,days_since_update,ctr_prior90
40261,content_1f49061d99a764db,client_23a62021009f63c4,0.885839,zero_click_risk,review_for_refresh,854,0,1.119058,166,0.0
305389,content_ebf7d5b38ea54c9a,client_23a62021009f63c4,0.885818,zero_click_risk,review_for_refresh,1231,0,0.929008,216,0.0
319320,content_f6b8c3a11ade2d01,client_23a62021009f63c4,0.883647,zero_click_risk,review_for_refresh,1473,0,0.983521,74,0.0
149312,content_738f3a26084f61b2,client_23a62021009f63c4,0.883337,zero_click_risk,review_for_refresh,1509,0,1.084972,81,0.0
246298,content_be769860a6f684d9,client_23a62021009f63c4,0.883118,zero_click_risk,review_for_refresh,1798,0,0.345532,78,0.0
169503,content_83197c88983ff2a4,client_23a62021009f63c4,0.883036,zero_click_risk,review_for_refresh,1628,0,1.076238,81,0.0
286059,content_dd1046faf497a3fe,client_23a62021009f63c4,0.882935,zero_click_risk,review_for_refresh,1597,0,0.765488,81,0.0
245338,content_bdc0e20e9e04419d,client_23a62021009f63c4,0.882883,zero_click_risk,review_for_refresh,1523,0,1.318468,74,0.0
247739,content_bf97585c2629d43f,client_23a62021009f63c4,0.882732,zero_click_risk,review_for_refresh,813,0,1.849893,159,0.0
67788,content_34b2d480f97810ba,client_23a62021009f63c4,0.882389,zero_click_risk,review_for_refresh,705,0,1.477922,159,0.0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Who: a FlyRank content reviewer with limited weekly review capacity, deciding which
pages to open first -- not an automated system that edits pages on its own.

What it's for: prioritizing attention, nothing more. It is decision-support, not a
verdict.

Where it stops being valid: outside the client cohort and month this was validated
on (one mid-panel month, 47 total clients). It has not been tested on the sealed
final month, on clients with under 90 days of tracking history, or on any inventory
outside this warehouse release. It should not be used to auto-trigger content
changes, and it says nothing about causality -- a flagged page refreshing and
recovering is not proven by this data.

In [9]:
# no numbers needed here -- this section is a scope statement, not a computation
print("validated cohort: 47 total clients (37 train + 10 test), month=2026-03 development window")

validated cohort: 47 total clients (37 train + 10 test), month=2026-03 development window


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

What a reviewer must check before acting on any row: does the page's actual content
still match its original intent (a stale page might have become irrelevant, not
just old), is the client still active/under contract, and does the page carry any
legal, medical, or compliance sensitivity that a generic refresh could get wrong.

The no-go list -- never automate:
- Never auto-publish a refresh without human review of the actual page content.
- Never treat "zero_click_risk" as proof the content itself is bad -- it could be a
  title/meta problem, a targeting problem, or genuinely fine content nobody's
  finding yet.
- Never use this queue to justify removing a page without a human confirming it's
  genuinely low-value, not just low-signal in this dataset.
- Never extend these scores to a client outside the validated cohort without
  re-checking that client's own data quality first.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Signs this playbook has gone stale and needs a re-check, not blind trust:

- The label's positive rate drifts far from the ~16-37% range seen in development --
  a sudden jump back toward 90%+ would suggest the same day-level noise bug
  resurfacing in a different form.
- The gap between Random Forest AUC and the rule baseline AUC shrinks toward zero on
  a fresh month -- would mean the model stopped adding real value over the simple
  rule.
- A new month's feature distributions (impressions, CTR) shift noticeably from
  this development month's ranges -- the model was only validated on this one
  slice's typical values.
- Retrain trigger: re-run this whole pipeline at least once against the sealed
  final month before trusting it beyond this one validated slice.

In [10]:
print("baseline AUC this run:", "recompute below if re-validating on a new month")
print("dev-month label positive rate for reference:", label_by_content["future_decline_label"].mean())

baseline AUC this run: recompute below if re-validating on a new month
dev-month label positive rate for reference: 0.16361831545155023


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Writing the queue and the reason-code breakdown to work/outputs/ so the capstone
paper can point to real, versioned files instead of numbers copied by hand.

In [11]:
import os
os.makedirs("work/outputs", exist_ok=True)

queue_export = queue[["content_hash_id", "client_hash_id", "rf_proba", "reason_code", "action"]]
queue_export.to_csv("work/outputs/action_playbook_queue.csv", index=False)

summary = {
    "reason_code_counts": queue["reason_code"].value_counts().to_dict(),
    "total_rows": len(queue),
    "label_positive_rate": float(label_by_content["future_decline_label"].mean()),
}
import json
with open("work/outputs/action_playbook_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("wrote work/outputs/action_playbook_queue.csv and action_playbook_summary.json")
print(summary)

wrote work/outputs/action_playbook_queue.csv and action_playbook_summary.json
{'reason_code_counts': {'no_action_needed': 16339, 'model_only': 8672, 'zero_click_risk': 4772, 'model_and_rule_agree': 2943}, 'total_rows': 32726, 'label_positive_rate': 0.16361831545155023}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.